### Libraries



In [ ]:
!pip install -q \
  datasets \
  transformers \
  huggingface_hub

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline, DataCollatorWithPadding
from peft import PeftModel
from huggingface_hub import hf_hub_download, login
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix
from collections import Counter
import json
import time
import psutil
import os

### Load model

In [ ]:
login()

In [ ]:
model_id = "eduhuemar001/distilbert-news"

#subfolder = "checkpoints/checkpoint-842"  # Epoch 1
subfolder = "checkpoints/checkpoint-1684"  # Epoch 2
#subfolder = "checkpoints/checkpoint-2526" # Epoch 3
#subfolder = "checkpoints/checkpoint-3368" # Epoch 4
#subfolder = "checkpoints/checkpoint-4210" # Epoch 5
#subfolder = "checkpoints/checkpoint-5052" # Epoch 6

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id, subfolder=subfolder, num_labels=2)
model.to("cuda")
model.eval()

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


### Download test data



In [ ]:
HF_DATASET_REPO = "eduhuemar001/news"

# Download JSON file from HF dataset repo
test_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename="test.json",
    repo_type="dataset",
    local_dir=".",
    local_dir_use_symlinks=False
)

# Load dataset
with open(test_path, "r", encoding="utf-8") as f:
    dataset_test = json.load(f)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


### Tokenization

In [ ]:
class NewsPairDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.data = data                  # list of dicts with keys of title, text, status
        self.tok = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        item = self.data[i]
        title = (item.get("title") or "").strip()
        text  = (item.get("text")  or "").strip()
        label = int(item.get("status"))   # 0/1

        enc = self.tok(
            text=title,                   # News title
            text_pair=text,               # News article
            truncation="only_second",     # keep full title, truncate only news article
            max_length=self.max_length,
            padding=False,                # let collator pad
            return_attention_mask=True
        )

        out = {
            "input_ids": torch.tensor(enc["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(label, dtype=torch.long),
        }
        return out

test_tokenized = NewsPairDataset(dataset_test, tokenizer, max_length=512)

### Testing function

In [ ]:
def evaluate_model(model, tokenizer, token_ds, batch_size=32, device="cuda"):
    collator = DataCollatorWithPadding(tokenizer)
    loader = DataLoader(token_ds, batch_size=batch_size, shuffle=False, collate_fn=collator)

    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating", leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop("labels")
            outputs = model(**batch)
            preds = torch.argmax(outputs.logits, dim=-1)

            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())

    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_labels).numpy()

    print("Classification report:")
    print(classification_report(
        y_true, y_pred,
        digits=3,
        zero_division=0
    ))

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    cm_df = pd.DataFrame(cm,
                      index=["true_fake", "true_real"],
                      columns=["pred_fake", "pred_real"])
    print("\nConfusion matrix:")
    print(cm_df)

    return {
        "report": classification_report(
            y_true, y_pred,
            digits=3,
            zero_division=0,
            output_dict=True
        ),
        "confusion_matrix": cm,
    }

In [ ]:
def _to_plain(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, dict):
        return {k: _to_plain(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_to_plain(x) for x in obj]
    return obj

def upload_metrics(
    metrics: dict,
    file_name: str,
    subdir: str = "testing",
    repo_id: str = "eduhuemar001/distilbert-news",
    repo_type: str = "model",
):
    os.makedirs("testing_local", exist_ok=True)
    local_path = os.path.join("testing_local", file_name)

    metrics_clean = _to_plain(metrics)

    with open(local_path, "w", encoding="utf-8") as f:
        json.dump(metrics_clean, f, ensure_ascii=False, indent=2)

    api = HfApi()
    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=f"{subdir}/{file_name}",
        repo_id=repo_id,
        repo_type=repo_type,
        commit_message=f"Add evaluation metrics {file_name}"
    )
    print(f"Uploaded to https://huggingface.co/{repo_id}/tree/main/{subdir}")

### Testing for Epoch 1



In [ ]:
metrics = evaluate_model(model, tokenizer, test_tokenized)
upload_metrics(metrics, "epoch_1_metrics.json")

Evaluating:   0%|          | 0/281 [00:00<?, ?it/s]

Classification report:
              precision    recall  f1-score   support

           0      0.998     0.990     0.994      4697
           1      0.989     0.998     0.993      4283

    accuracy                          0.994      8980
   macro avg      0.993     0.994     0.994      8980
weighted avg      0.994     0.994     0.994      8980


Confusion matrix:
           pred_fake  pred_real
true_fake       4648         49
true_real          9       4274
Uploaded to https://huggingface.co/eduhuemar001/distilbert-news/tree/main/testing


### Testing for Epoch 2



In [ ]:
evaluate_model(model, tokenizer, test_tokenized)

### Testing for Epoch 3



In [ ]:
evaluate_model(model, tokenizer, test_tokenized)

### Testing for Epoch 4





In [ ]:
evaluate_model(model, tokenizer, test_tokenized)

### Testing for Epoch 5



In [ ]:
evaluate_model(model, tokenizer, test_tokenized)

### Testing for Epoch 6



In [ ]:
evaluate_model(model, tokenizer, test_tokenized)